# Prepare CIFAR-10 EmbeddingGemma 2 vectors

Download CIFAR-10 and encode a balanced sample of its training images directly with Google’s `google/embeddinggemma-2` model. This writes normalized 768-dimensional vectors, labels, source indices, and a manifest under `data/cifar10/cache/` for the comparison notebook.

Install dependencies in the active kernel if needed: `%pip install -U sentence-transformers transformers torchvision pillow`. The model weights and CIFAR-10 data download on first run. EmbeddingGemma 2 accepts images directly and maps them to its shared vector space ([Google multimodal guide](https://ai.google.dev/gemma/docs/embeddinggemma/multimodal-embeddinggemma-with-sentence-transformers)).

In [1]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "imste" / "__init__.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate this IMSTE repository from the notebook working directory.")
DATA_DIR = REPO_ROOT / "data" / "cifar10"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_ROOT)
print("CIFAR-10 data:", DATA_DIR)


Repository: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree
CIFAR-10 data: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar10


## Sampling and encoder settings

The default run selects 1,000 images from each of the ten classes (10,000 total). Set `SAMPLES_PER_CLASS = 5_000` to encode all 50,000 training images. The encoder’s unused audio weights are disabled; the image encoder remains enabled.

In [2]:
import hashlib
import json
import time

import numpy as np
import torch
from PIL import Image
from torchvision.datasets import CIFAR10

MODEL_ID = "google/embeddinggemma-2"
RANDOM_STATE = 42
SAMPLES_PER_CLASS = 1000  # Set to 5_000 to embed all 50,000 training images.
ENCODER_BATCH_SIZE = 32
FORCE_REEMBED = False

if torch.cuda.is_available():
    DEVICE = "cuda"
    TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    TORCH_DTYPE = torch.float32  # EmbeddingGemma 2 does not support float16 activations.
else:
    DEVICE = "cpu"
    TORCH_DTYPE = torch.float32

MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1].replace("-", "_")
EMBEDDING_CACHE = CACHE_DIR / (
    f"cifar10_train_{SAMPLES_PER_CLASS}_per_class_seed{RANDOM_STATE}_{MODEL_TAG}.npz"
)
MANIFEST_PATH = CACHE_DIR / "embeddinggemma2_cifar10_manifest.json"
print(f"Embedding model: {MODEL_ID} | device={DEVICE} | dtype={TORCH_DTYPE}")
print(f"Balanced sample: {SAMPLES_PER_CLASS:,} images per class")


Embedding model: google/embeddinggemma-2 | device=mps | dtype=torch.float32
Balanced sample: 1,000 images per class


In [3]:
from tqdm.auto import tqdm

train_data = CIFAR10(root=str(DATA_DIR / "raw"), train=True, download=True)
class_names = list(train_data.classes)
all_targets = np.asarray(train_data.targets, dtype=np.int64)

if not 1 <= SAMPLES_PER_CLASS <= 5_000:
    raise ValueError("SAMPLES_PER_CLASS must be between 1 and 5,000.")

if EMBEDDING_CACHE.is_file() and not FORCE_REEMBED:
    with np.load(EMBEDDING_CACHE, allow_pickle=False) as saved:
        X = saved["embeddings"]
        labels = saved["labels"]
        sample_indices = saved["indices"]
    print("Loaded cached image vectors:", EMBEDDING_CACHE)
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sample_indices = np.concatenate([
        rng.choice(np.flatnonzero(all_targets == class_id), SAMPLES_PER_CLASS, replace=False)
        for class_id in range(len(class_names))
    ])
    rng.shuffle(sample_indices)
    labels = all_targets[sample_indices]

    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(
        MODEL_ID,
        device=DEVICE,
        model_kwargs={"torch_dtype": TORCH_DTYPE},
        config_kwargs={"audio_config": None},
    )
    X = np.empty((len(sample_indices), 768), dtype=np.float32)
    started = time.perf_counter()
    for start in tqdm(range(0, len(sample_indices), ENCODER_BATCH_SIZE), desc="Embedding CIFAR-10 images"):
        batch_indices = sample_indices[start : start + ENCODER_BATCH_SIZE]
        image_inputs = [
            {"image": Image.fromarray(train_data.data[int(index)]).convert("RGB")}
            for index in batch_indices
        ]
        batch_vectors = model.encode(
            image_inputs,
            batch_size=len(image_inputs),
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        X[start : start + len(batch_indices)] = batch_vectors
    embedding_seconds = time.perf_counter() - started
    np.savez_compressed(
        EMBEDDING_CACHE,
        embeddings=X,
        labels=labels,
        indices=sample_indices,
        class_names=np.asarray(class_names),
        model_id=np.asarray(MODEL_ID),
    )
    print(f"Encoded {len(X):,} images in {embedding_seconds / 60:.1f} min")
    print("Saved image vectors:", EMBEDDING_CACHE)

vector_fingerprint = hashlib.sha256(np.ascontiguousarray(X).view(np.uint8)).hexdigest()
manifest = {
    "dataset": "CIFAR-10 train",
    "model_id": MODEL_ID,
    "samples_per_class": SAMPLES_PER_CLASS,
    "random_state": RANDOM_STATE,
    "embedding_file": EMBEDDING_CACHE.name,
    "shape": list(X.shape),
    "dtype": str(X.dtype),
    "vector_sha256": vector_fingerprint,
    "class_names": class_names,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")

print("Embedding matrix:", X.shape, X.dtype)
print(f"Finite values: {np.isfinite(X).all()} | mean L2 norm: {np.linalg.norm(X, axis=1).mean():.4f}")
print("Class counts:", {class_names[i]: int(np.sum(labels == i)) for i in range(len(class_names))})
print("Manifest:", MANIFEST_PATH)


W1113 07:43:27.995000 37420 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/624 [00:00<?, ?it/s]

Embedding CIFAR-10 images:   0%|          | 0/313 [00:00<?, ?it/s]

Encoded 10,000 images in 46.0 min
Saved image vectors: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar10/cache/cifar10_train_1000_per_class_seed42_embeddinggemma_2.npz
Embedding matrix: (10000, 768) float32
Finite values: True | mean L2 norm: 1.0000
Class counts: {'airplane': 1000, 'automobile': 1000, 'bird': 1000, 'cat': 1000, 'deer': 1000, 'dog': 1000, 'frog': 1000, 'horse': 1000, 'ship': 1000, 'truck': 1000}
Manifest: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar10/cache/embeddinggemma2_cifar10_manifest.json
